# 01 — Data lake inventory

**Phase 1: what have I actually got?**

A tally of the lake — databases, tables, row counts, partition ranges, date
coverage, site and circuit counts — and the **real** schema of the candidate
tables, read from the catalogue rather than inferred from what the pipeline code
implies they contain.

---

### Expected Athena scan

**Effectively zero bytes of DATA.** Everything below reads metadata:

| Source | What it gives | Data scanned |
|---|---|---|
| Glue API | every table, its format, location, columns, partition keys | none — not an Athena query |
| `information_schema.columns` | the schema Athena actually resolves | none |
| `"table$partitions"` (Iceberg) | **exact per-partition row counts and byte sizes** | manifests only, ~0 |
| dimension tables | site and circuit counts | a few MB |
| partition-filtered `LIMIT 5` | what a row looks like | one partition, a handful of columns |

The one that matters is `$partitions`. On an Iceberg table it returns exact row
counts and file sizes straight from the manifests, so the entire "how big is `ts`"
question is answered without reading a single data file. If `ts` turns out to be
Hive rather than Iceberg, `$partitions` gives partition *values* only and the row
counts have to be bought — the notebook says so where that happens and does not
buy them silently.

Athena's 10 MB per-query minimum dominates: roughly 25 queries × 10 MB ≈ **250 MB
billable, about AUD 0.002.** Section 9 prints the measured figure.

**Nothing in this notebook scans more than a few GB. If any query does, the guard
in `ami_athena` has been bypassed and that is a bug.**


## Setup


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_current, *_current.parents) if (p / "bms_sa_review").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from bms_sa_review.ami_data_analysis.config import ami_config as C
from bms_sa_review.ami_data_analysis.lib import ami_athena as A
from bms_sa_review.ami_data_analysis.lib import ami_inventory as I

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 220)

A.reset_scan_log()
A.require_credentials()
print("Credentials OK. Starting inventory.")


## 1. The catalogue — every table in every database

Glue API, free. This is the ground truth for *what exists*, as distinct from what
the pipeline code happens to reference.

`is_iceberg` is the column to read first. An Iceberg table exposes `$partitions`,
`$files`, `$snapshots` and `$history`, which is how the next section gets row
counts for nothing. A Hive table does not.


In [ ]:
catalog = I.glue_inventory()
print(f"{len(catalog)} tables across {catalog.database.nunique()} databases")
display(I.database_summary(catalog))


In [ ]:
display(catalog[["database", "table", "table_type", "format", "is_iceberg",
                 "n_columns", "partition_keys", "updated"]])


### Storage locations

Where the files physically are. Two tables pointing at the same prefix, or a table
whose location is somewhere unexpected, is worth noticing now rather than in Phase 4.


In [ ]:
display(catalog[["database", "table", "location"]].sort_values(["location"])
        .reset_index(drop=True))


## 2. Real schemas

`information_schema.columns` is what **Athena** will let you select. That is not
always what Glue declares — `aws_config.describe()` notes that the Glue column
list can come back empty for Iceberg tables, which is why `n_columns` above may
disagree with what you see here.

One query per database rather than one `DESCRIBE` per table.


In [ ]:
columns = pd.concat(
    [I.column_inventory(db).assign(database=db) for db in (C.SA, C.SAI)],
    ignore_index=True,
)
print(f"{len(columns):,} column definitions across {columns.table_name.nunique()} tables")

width = (columns.groupby(["database", "table_name"], as_index=False)
                .agg(n_columns=("column_name", "count")))
display(width.sort_values("n_columns", ascending=False).reset_index(drop=True))


## 3. Partition metadata — the free row counts

For every table that declares partition keys, read its `$partitions` metadata
table. On Iceberg this returns exact record counts, file counts and total bytes
per partition, from the manifests, without touching data.

Tables that will not serve `$partitions` are logged and skipped — one awkward
table should not stop the inventory.


In [ ]:
totals, raw_partitions, partition_log = I.probe_partitions(catalog)
print()
display(partition_log)


### What `$partitions` actually returned

Shown raw, before anything interprets it. The column set differs between Athena
engine versions and between Iceberg and Hive, and `normalise_partitions()` is
written to be tolerant of that — but it is worth seeing the real shape once,
because Phase 4's size estimate is built on these columns.


In [ ]:
ts_key = next((k for k in raw_partitions if k.endswith(".ts")), None)
if ts_key is None:
    print("No `ts` partition metadata was returned. Available keys:")
    for key in sorted(raw_partitions):
        print("  -", key)
else:
    print(f"raw {ts_key}$partitions -- {len(raw_partitions[ts_key])} rows, "
          f"columns: {list(raw_partitions[ts_key].columns)}")
    display(raw_partitions[ts_key].head(10))


## 4. The one-screen summary

Everything above, joined. Sorted by row count, so the fact tables sort to the top
and the dimension tables fall to the bottom.

Blank `n_rows` means the table was not probed (unpartitioned) or `$partitions`
carried no counts (Hive) — **not** that the table is empty.


In [ ]:
summary = I.summary_table(catalog, totals)
display(summary)


## 5. `ts` in detail — and the `is_pv` question

`ts` is partitioned on `(year, month, is_pv)`. That third key is the one this
whole exercise turns on.

Every query in the Stage 1 pipeline filters `is_pv = True`. If `ts` has no
`is_pv = False` partitions, or they are near-empty, then the load circuits a
synthetic meter needs do not exist in this table and Phase 2 has a very different
decision to make. If they are substantial, the raw circuit data is there.

**This cell answers that, for free, from the manifests.** It is the most
consequential number in the notebook.


In [ ]:
ts_tidy = pd.DataFrame()
if ts_key is not None:
    ts_tidy = I.normalise_partitions(raw_partitions[ts_key])
    display(ts_tidy)

    if "is_pv" in ts_tidy.columns and "n_rows" in ts_tidy.columns:
        by_is_pv = (ts_tidy.groupby("is_pv", as_index=False)
                           .agg(n_partitions=("is_pv", "count"),
                                n_rows=("n_rows", "sum"),
                                size_bytes=("size_bytes", "sum")))
        by_is_pv["size"] = by_is_pv.size_bytes.map(A.fmt_bytes)
        by_is_pv["share_of_rows"] = (
            by_is_pv.n_rows / by_is_pv.n_rows.sum()
        ).map(lambda v: f"{v:.1%}")
        print("ts rows by is_pv partition:")
        display(by_is_pv[["is_pv", "n_partitions", "n_rows", "size", "share_of_rows"]])
    else:
        print("`$partitions` did not carry an is_pv breakdown with row counts.")
        print("Columns available:", list(ts_tidy.columns))
        print("Phase 2 will have to establish is_pv coverage another way.")


### Coverage over time

Rows per month, so gaps and the usable date range are visible rather than assumed.


In [ ]:
if len(ts_tidy) and {"year", "month"} <= set(ts_tidy.columns):
    by_month = (ts_tidy.dropna(subset=["year", "month"])
                       .groupby(["year", "month"], as_index=False)
                       .agg(n_rows=("n_rows", "sum") if "n_rows" in ts_tidy.columns
                                     else ("month", "count"),
                            n_partitions=("month", "count")))
    by_month["ym"] = (by_month.year.astype(int).astype(str) + "-"
                      + by_month.month.astype(int).astype(str).str.zfill(2))
    display(by_month[["ym", "n_partitions", "n_rows"]])
    print(f"Coverage: {by_month.ym.iloc[0]} .. {by_month.ym.iloc[-1]} "
          f"({len(by_month)} month-partitions)")
else:
    print("No year/month partition metadata available for ts.")


## 6. Candidate tables — schema and a real sample

The shortlist is a naming heuristic and nothing more; it orders this section, it
does not exclude anything. Everything is still in the summary table above.

For each candidate: the schema Athena resolves, then partition coverage. Samples
of the large tables come in section 7, where a partition predicate can be chosen
from what section 5 found.


In [ ]:
shortlisted = I.guess_candidates(catalog)
candidates = shortlisted[shortlisted.shortlisted].reset_index(drop=True)
print(f"{len(candidates)} of {len(catalog)} tables shortlisted by name.")
display(candidates[["database", "table", "is_iceberg", "n_columns", "partition_keys"]])


In [ ]:
for entry in candidates.itertuples(index=False):
    key = f"{entry.database}.{entry.table}"
    schema = columns[(columns.database == entry.database)
                     & (columns.table_name == entry.table)]
    print("=" * 78)
    print(f"{key}   [{'iceberg' if entry.is_iceberg else 'hive'}]"
          f"   partitioned by: {entry.partition_keys or '(none)'}")
    stats = totals.get(key, {})
    if stats.get("n_partitions"):
        print(f"  {stats['n_partitions']} partitions"
              + (f", {stats['n_rows']:,.0f} rows" if stats.get("n_rows") else "")
              + (f", {A.fmt_bytes(stats['size_bytes'])}" if stats.get("size_bytes") else "")
              + (f", {stats.get('first_partition')} .. {stats.get('last_partition')}"
                 if stats.get("first_partition") else ""))
    if len(schema):
        display(schema[["ordinal_position", "column_name", "data_type"]]
                .reset_index(drop=True))
    else:
        print("  (no columns resolved via information_schema)")


## 7. Sample rows

The large tables get a partition predicate taken from the coverage found above,
so this reads one month rather than the table. The dimension tables are small
enough to read directly.

`meta_up23c` is the circuit dimension — one row per circuit. Its column list is
what Phase 3 will build the circuit-to-signal mapping from, so it is worth
reading carefully here even though the taxonomy itself is Phase 3's job.


In [ ]:
latest_year = latest_month = None
if len(ts_tidy) and {"year", "month"} <= set(ts_tidy.columns):
    ordered = ts_tidy.dropna(subset=["year", "month"]).sort_values(["year", "month"])
    latest_year = int(ordered.year.iloc[-1])
    latest_month = int(ordered.month.iloc[-1])
    print(f"Sampling ts from the newest partition: year={latest_year} month={latest_month}")
else:
    print("No partition metadata for ts -- skipping the ts sample rather than guessing.")


In [ ]:
if latest_year is not None:
    ts_sample = A.aq(
        f"""
        SELECT *
        FROM ts
        WHERE year = {latest_year} AND month = {latest_month}
        LIMIT 5
        """,
        database=C.SAI,
        label=f"ts sample {latest_year}-{latest_month:02d}",
    )
    display(ts_sample)
    print("Columns:", list(ts_sample.columns))


In [ ]:
meta_sample = A.aq("SELECT * FROM meta_up23c LIMIT 5", database=C.SAI,
                   label="meta_up23c LIMIT 5")
display(meta_sample)
print("meta_up23c columns:", list(meta_sample.columns))


In [ ]:
fleet = I.dimension_counts("meta_up23c", database=C.SAI)
display(fleet)

for table in ("circuits", "sites"):
    try:
        display(A.aq(f"SELECT * FROM {table} LIMIT 5", database=C.SA,
                     label=f"{table} LIMIT 5"))
    except Exception as exc:
        print(f"{table}: {type(exc).__name__}: {str(exc)[:200]}")


In [ ]:
# `partition_lookup` is the pipeline's own record of which partitions exist.
# Worth cross-checking against the $partitions metadata above -- a disagreement
# means one of them is stale.
try:
    lookup = A.aq("SELECT * FROM partition_lookup", database=C.SA,
                  label="partition_lookup")
    print(f"{len(lookup)} rows")
    display(lookup.head(30))
except Exception as exc:
    print(f"partition_lookup: {type(exc).__name__}: {str(exc)[:200]}")


## 8. `structured_data` — the Phase 2 candidate

`ciccada_config.TABLES` maps the logical name to the rebuilt table. Phase 2 decides
whether to build from this or from raw `ts`; this section just establishes what it
is and how big.

Note what `build_structured_data.py` does: it aggregates circuits to **site** level
and filters `is_pv = True` throughout. If that filter has already discarded the
load circuits, this table cannot be the source. **Phase 2 checks that properly** —
here we only record the table's shape.


In [ ]:
structured = C.TABLES["structured_data"]
print(f"structured_data -> {structured}"
      f"   (rebuilt: {'structured_data' in C.REBUILT})")

structured_key = f"{C.SAI}.{structured}"
if structured_key in totals:
    print(totals[structured_key])

structured_schema = columns[columns.table_name == structured]
display(structured_schema[["database", "ordinal_position", "column_name", "data_type"]]
        .reset_index(drop=True))


## 9. What this cost

Measured, not estimated. `source = unavailable` means the figure could not be
recovered from the Athena response — the query still ran and was still billed, so
the total is a lower bound in that case.


In [ ]:
display(A.scan_report())

## 10. Inventory and read on candidates

*To be written once the output above has been read. Deliberately left empty rather
than pre-filled with a guess — the point of this notebook is that the summary
comes from what the catalogue actually says.*

**Inventory**

- Databases and tables:
- Fact tables and their grain:
- Coverage:
- `ts` `is_pv` split:
- Fleet size (sites / circuits):

**Live candidates for a synthetic AMI dataset**

1.
2.

**Ruled out, and why**

-

**Carried into Phase 2**

-
